In [ ]:
# Tune Elastic Net on walk-forward validation folds and freeze an out-of-fold threshold.
# Evaluate the chronological test split only after the model and threshold are fixed.

In [2]:
import sys as _sys
from pathlib import Path as _Path
_UTILS_ROOT = next(path for path in (_Path.cwd().resolve(), *_Path.cwd().resolve().parents)
                   if (path / "notebook_utils.py").is_file())
if str(_UTILS_ROOT) not in _sys.path: _sys.path.insert(0, str(_UTILS_ROOT))
from notebook_utils import project_paths, file_hash, probability_metrics, calibration_table, load_split
from pathlib import Path
import html, inspect, json, platform, time, warnings, joblib
import numpy as np
import pandas as pd
import sklearn
import statsmodels.api as sm
from scipy import sparse
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_selection import VarianceThreshold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, precision_recall_curve, roc_auc_score, roc_curve
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "data/processed/train_matrix.csv").exists(): ROOT = ROOT.parent
PROCESSED = project_paths(ROOT).processed
REPORTS = ROOT / "artifacts/19_model_elastic_net_logistic"
MODELS = project_paths(ROOT).models
C_GRID, L1_RATIOS = [.001, .01, .1, 1., 10.], [.1, .5, .9]
CLASS_WEIGHTS = {"unweighted": None, "balanced": "balanced"}
MAX_ITER, TOL, SEED = 12000, 1e-4, 42
input_hashes = {}

def make_classifier(C, l1_ratio, class_weight):
    params = dict(C=C, l1_ratio=l1_ratio, class_weight=class_weight, solver="saga",
                  max_iter=MAX_ITER, tol=TOL, random_state=SEED, warm_start=False)
    if inspect.signature(LogisticRegression).parameters["penalty"].default != "deprecated": params["penalty"] = "elasticnet"
    return LogisticRegression(**params)

def calibration_fit(labels, probabilities):
    clipped = np.clip(probabilities, 1e-6, 1 - 1e-6)
    logits = np.log(clipped / (1 - clipped))
    try:
        fit = sm.GLM(labels, sm.add_constant(logits, has_constant="add"), family=sm.families.Binomial()).fit(maxiter=100, disp=0)
        return float(fit.params[0]), float(fit.params[1]), "ok"
    except Exception as exc:
        return np.nan, np.nan, type(exc).__name__

def top_risk(labels, probabilities, fraction):
    count = max(1, int(np.ceil(len(labels) * fraction)))
    indices = np.argsort(-probabilities, kind="stable")[:count]
    cases, total = int(labels[indices].sum()), int(np.asarray(labels).sum())
    return count, cases, cases / total if total else np.nan, cases / count

artifact_path = MODELS / "preprocessing_pipeline.joblib"
input_hashes[artifact_path] = file_hash(artifact_path)
preprocessing = joblib.load(artifact_path)
schema = preprocessing.project_schema_
if schema.get("specification") != "B" or schema.get("fit_split") != "train":
    raise ValueError("Elastic Net requires the training-fitted Spec B preprocessing pipeline")
FEATURES, ID_COLUMNS = schema["matrix_feature_columns"], schema["matrix_metadata_columns"]
KEYS = ["company_id", "prediction_date"]


In [3]:
# Tune each candidate on every walk-forward training/validation pair.
fold_ids_path = PROCESSED / "walk_forward_fold_ids.csv"
input_hashes[fold_ids_path] = file_hash(fold_ids_path)
manifest_columns = ["fold", "split", "company_id", "prediction_date", "metadata_event_id", "company_group_id", "metadata_outcome_end_date"]
manifest = pd.read_csv(fold_ids_path, usecols=manifest_columns,
    dtype={"company_id":"string", "metadata_event_id":"string", "company_group_id":"string"})
manifest["prediction_date"] = pd.to_datetime(manifest.prediction_date, errors="raise")
manifest["metadata_outcome_end_date"] = pd.to_datetime(manifest.metadata_outcome_end_date, errors="raise")
fold_names = sorted(manifest.loc[manifest.fold.ne("final_test"), "fold"].unique())
if not fold_names: raise ValueError("No walk-forward validation folds found")
fold_data = {}
for fold in fold_names:
    members = manifest.loc[manifest.fold.eq(fold)]
    train_ids, val_ids = members.loc[members.split.eq("train")], members.loc[members.split.eq("validation")]
    if train_ids.empty or val_ids.empty: raise ValueError(f"{fold}: empty training or validation split")
    if set(train_ids.company_group_id).intersection(val_ids.company_group_id): raise ValueError(f"{fold}: company group crosses split")
    if not train_ids.metadata_outcome_end_date.max() < val_ids.prediction_date.min(): raise ValueError(f"{fold}: training outcome reaches validation")
    pipeline_path = MODELS / "walk_forward" / f"{fold}_spec_b_preprocessing_pipeline.joblib"
    train_path = PROCESSED / "walk_forward" / fold / "spec_b" / "train_matrix.csv"
    val_path = PROCESSED / "walk_forward" / fold / "spec_b" / "validation_matrix.csv"
    for path in [pipeline_path, train_path, val_path]: input_hashes[path] = file_hash(path)
    fold_pipeline = joblib.load(pipeline_path)
    fold_schema = fold_pipeline.project_schema_
    if fold_schema.get("fit_split") != f"{fold}/train" or fold_schema.get("specification") != "B": raise ValueError(f"{fold}: wrong fold pipeline")
    if fold_schema["training_event_ids"] != train_ids.metadata_event_id.tolist(): raise ValueError(f"{fold}: training IDs differ from pipeline provenance")
    train_frame = pd.read_csv(train_path, dtype={"company_id":"string", "metadata_event_id":"string"})
    val_frame = pd.read_csv(val_path, dtype={"company_id":"string", "metadata_event_id":"string"})
    fold_features = fold_schema["matrix_feature_columns"]
    if train_frame.columns.tolist() != ID_COLUMNS + fold_features or val_frame.columns.tolist() != ID_COLUMNS + fold_features: raise ValueError(f"{fold}: matrix schema mismatch")
    if train_frame.metadata_event_id.tolist() != train_ids.metadata_event_id.tolist() or val_frame.metadata_event_id.tolist() != val_ids.metadata_event_id.tolist(): raise ValueError(f"{fold}: matrix membership/order mismatch")
    selector = VarianceThreshold(0.)
    Xtr = sparse.csr_matrix(selector.fit_transform(train_frame[fold_features].to_numpy(dtype=float)))
    Xva = sparse.csr_matrix(selector.transform(val_frame[fold_features].to_numpy(dtype=float)))
    fold_data[fold] = (train_frame, val_frame, Xtr, Xva, train_frame.target.to_numpy(dtype=int), val_frame.target.to_numpy(dtype=int))

fold_rows, candidate_oof = [], {}
for weight_name, class_weight in CLASS_WEIGHTS.items():
    for l1_ratio in L1_RATIOS:
        for C in C_GRID:
            candidate = f"{weight_name}_l1_{l1_ratio:g}_C_{C:g}"
            predictions_by_fold = {}
            for fold, (train_frame, val_frame, Xtr, Xva, ytr, yva) in fold_data.items():
                model = make_classifier(C, l1_ratio, class_weight)
                started = time.perf_counter()
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter("always")
                    model.fit(Xtr, ytr)
                warned = any(issubclass(w.category, ConvergenceWarning) for w in caught)
                eligible = bool(not warned and model.n_iter_.max() < MAX_ITER and np.isfinite(model.coef_).all())
                prob = model.predict_proba(Xva)[:, 1]
                fold_rows.append({"candidate_id":candidate, "fold":fold, "C":C, "l1_ratio":l1_ratio,
                    "class_weight":weight_name, "eligible":eligible, "iterations":int(model.n_iter_.max()),
                    "validation_pr_auc":average_precision_score(yva, prob), "validation_roc_auc":roc_auc_score(yva, prob),
                    "validation_brier_score":brier_score_loss(yva, prob), "elapsed_seconds":time.perf_counter()-started,
                    "warnings":"; ".join(sorted({str(w.message) for w in caught}))})
                if eligible:
                    predictions_by_fold[fold] = pd.DataFrame({"company_id":val_frame.company_id,
                        "prediction_date":val_frame.prediction_date, "metadata_event_id":val_frame.metadata_event_id,
                        "fold":fold, "target":yva, "bankruptcy_probability":prob})
            if len(predictions_by_fold) == len(fold_data): candidate_oof[candidate] = predictions_by_fold

fold_tuning = pd.DataFrame(fold_rows)
tuning = fold_tuning.groupby(["candidate_id","C","l1_ratio","class_weight"], as_index=False).agg(
    eligible=("eligible","all"), folds_completed=("fold","nunique"),
    mean_validation_pr_auc=("validation_pr_auc","mean"), mean_validation_roc_auc=("validation_roc_auc","mean"),
    mean_validation_brier_score=("validation_brier_score","mean"), total_iterations=("iterations","sum"))
tuning = tuning.loc[tuning.eligible].sort_values(
    ["mean_validation_pr_auc","mean_validation_roc_auc","mean_validation_brier_score","candidate_id"],
    ascending=[False,False,True,True], kind="stable").reset_index(drop=True)
if tuning.empty: raise RuntimeError("No candidate converged across all walk-forward folds")
best = tuning.iloc[0]
oof = pd.concat(list(candidate_oof[best.candidate_id].values()), ignore_index=True)
if oof.metadata_event_id.duplicated().any() or oof.target.nunique() != 2: raise ValueError("Pooled OOF predictions must have unique events and both classes")
oof = oof.sort_values(["prediction_date","company_id"], kind="stable").reset_index(drop=True)


In [4]:
# Freeze the threshold using pooled out-of-fold validation predictions.
y_oof, p_oof = oof.target.to_numpy(dtype=int), oof.bankruptcy_probability.to_numpy(dtype=float)
thresholds = np.unique(np.r_[0., .5, p_oof, np.nextafter(p_oof.max(), np.inf)])
threshold_rows = pd.DataFrame([probability_metrics(y_oof, p_oof, t) for t in thresholds])
threshold_rows["distance_from_half"] = (threshold_rows.threshold - .5).abs()
threshold_rows = threshold_rows.sort_values(
    ["balanced_accuracy","sensitivity","specificity","distance_from_half","threshold"],
    ascending=[False,False,False,True,False], kind="stable").reset_index(drop=True)
OPERATING_THRESHOLD = float(threshold_rows.iloc[0].threshold)

model_configuration = {"selected_candidate":best.candidate_id, "C":float(best.C), "l1_ratio":float(best.l1_ratio),
    "class_weight":best.class_weight, "mean_validation_pr_auc":float(best.mean_validation_pr_auc),
    "walk_forward_folds":fold_names, "selection_metric":"mean validation PR-AUC",
    "operating_threshold":OPERATING_THRESHOLD, "threshold_source":"pooled out-of-fold validation predictions",
    "threshold_metric":"balanced accuracy with sensitivity/specificity tie-breaks", "fit_split":"static training only",
    "solver":"saga", "max_iter":MAX_ITER, "tol":TOL, "seed":SEED,
    "preprocessing_sha256":input_hashes[artifact_path], "oof_validation_event_ids":oof.metadata_event_id.tolist()}

# Fit the selected specification on the static training split only.
train_path, train_ids_path = PROCESSED / "train_matrix.csv", PROCESSED / "train_ids.csv"
for path in [train_path, train_ids_path]: input_hashes[path] = file_hash(path)
train = load_split("train", PROCESSED, schema, ID_COLUMNS, FEATURES, input_hashes)
model_configuration["training_event_ids"] = train.metadata_event_id.tolist()
selector = VarianceThreshold(0.)
X_train = sparse.csr_matrix(selector.fit_transform(train[FEATURES].to_numpy(dtype=float)))
y_train = train.target.to_numpy(dtype=int)
final_model = make_classifier(model_configuration["C"], model_configuration["l1_ratio"], CLASS_WEIGHTS[model_configuration["class_weight"]])
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    final_model.fit(X_train, y_train)
if any(issubclass(w.category, ConvergenceWarning) for w in caught) or final_model.n_iter_.max() >= MAX_ITER: raise RuntimeError("Final training fit did not converge")

# Open the test split only after hyperparameters and threshold are fixed.
test = load_split("test", PROCESSED, schema, ID_COLUMNS, FEATURES, input_hashes)
X_test = sparse.csr_matrix(selector.transform(test[FEATURES].to_numpy(dtype=float)))
y_test = test.target.to_numpy(dtype=int)
p_test, p_train = final_model.predict_proba(X_test)[:,1], final_model.predict_proba(X_train)[:,1]

def summarize(split, labels, probabilities):
    m = probability_metrics(labels, probabilities, OPERATING_THRESHOLD)
    intercept, slope, status = calibration_fit(labels, probabilities)
    row = {"split":split, **m, "recall":m["sensitivity"],
           "precision":m["tp"]/(m["tp"]+m["fp"]) if m["tp"]+m["fp"] else 0.,
           "calibration_intercept":intercept, "calibration_slope":slope, "calibration_status":status}
    for f in [.01,.05,.10]:
        n,c,capture,precision=top_risk(labels,probabilities,f); tag=f"{int(f*100)}pct"
        row[f"top_risk_rows_{tag}"],row[f"top_risk_cases_{tag}"]=n,c
        row[f"top_risk_capture_{tag}"],row[f"top_risk_precision_{tag}"]=capture,precision
    return row

metrics = pd.DataFrame([summarize("walk_forward_oof_validation",y_oof,p_oof), summarize("final_test",y_test,p_test)])
train_metrics = summarize("training_diagnostic",y_train,p_train)
oof_predictions = oof.copy()
oof_predictions["predicted_target"] = (p_oof >= OPERATING_THRESHOLD).astype(int)
oof_predictions["prediction_split"] = "walk_forward_oof_validation"
test_predictions = test[ID_COLUMNS].copy()
test_predictions["prediction_probability"] = p_test
test_predictions["predicted_target"] = (p_test >= OPERATING_THRESHOLD).astype(int)
test_predictions["prediction_split"] = "final_test"
predictions = pd.concat([oof_predictions.rename(columns={"bankruptcy_probability":"prediction_probability"}), test_predictions], ignore_index=True)

active = selector.get_support()
active_features = np.asarray(FEATURES)[active].tolist()
coef = np.zeros(len(FEATURES)); coef[active] = final_model.coef_[0]
coefficients = pd.DataFrame({"feature":FEATURES,"included_in_model":active,"coefficient":coef,"absolute_coefficient":np.abs(coef),"nonzero":active & (coef!=0)})
coefficients = coefficients.sort_values(["absolute_coefficient","feature"],ascending=[False,True],kind="stable")
final_pipeline = Pipeline([("preprocessing",preprocessing),("training_constant_selector",selector),("classifier",final_model)])
model_configuration.update({"active_predictors":len(active_features),"training_constant_predictors":[f for f in FEATURES if f not in active_features],
    "training_event_ids":train.metadata_event_id.tolist(),"test_event_ids":test.metadata_event_id.tolist(),
    "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,"numpy":np.__version__,"pandas":pd.__version__,"joblib":joblib.__version__}})
final_pipeline.project_model_ = dict(model_configuration)
final_pipeline.project_model_["matrix_feature_columns"] = FEATURES
final_pipeline.project_model_["active_feature_columns"] = active_features
final_pipeline.project_model_["intercept"] = float(final_model.intercept_[0])

# Plot discrimination and calibration for OOF validation and final test.
REPORTS.mkdir(parents=True, exist_ok=True)
fig,axes=plt.subplots(1,2,figsize=(12,5),constrained_layout=True)
for labels,prob,name,color in [(y_oof,p_oof,"Walk-forward OOF validation","#d98300"),(y_test,p_test,"Final test","#1769aa")]:
    fpr,tpr,_=roc_curve(labels,prob); pr,rec,_=precision_recall_curve(labels,prob)
    axes[0].plot(fpr,tpr,color=color,label=f"{name}: {roc_auc_score(labels,prob):.3f}")
    axes[1].plot(rec,pr,color=color,label=f"{name}: {average_precision_score(labels,prob):.3f}")
axes[0].plot([0,1],[0,1],"--",color="black"); axes[0].set(xlabel="False positive rate",ylabel="Recall",title="ROC",xlim=(0,1),ylim=(0,1.02))
axes[1].set(xlabel="Recall",ylabel="Precision",title="Precision-recall",xlim=(0,1),ylim=(0,1.02))
for ax in axes: ax.legend(fontsize=8); ax.grid(alpha=.2)
fig.suptitle("Elastic Net selected by mean walk-forward PR-AUC")
performance_figure=REPORTS/"elastic_net_walk_forward_performance.png"; fig.savefig(performance_figure,dpi=180,bbox_inches="tight"); plt.close(fig)
calibration_tables=[]; fig,ax=plt.subplots(figsize=(6.5,5.5),constrained_layout=True)
for labels,prob,name,color in [(y_oof,p_oof,"Walk-forward OOF validation","#d98300"),(y_test,p_test,"Final test","#1769aa")]:
    tab=calibration_table(labels,prob); tab["split"]=name; calibration_tables.append(tab)
    ax.plot(tab.mean_predicted_probability,tab.observed_bankruptcy_rate,marker="o",color=color,label=name)
ax.plot([0,1],[0,1],"--",color="black",label="Perfect calibration")
ax.set(xlabel="Mean predicted relative-risk score",ylabel="Observed bankruptcy rate",title="Reliability diagram",xlim=(-.02,1.02),ylim=(-.02,1.02)); ax.legend(fontsize=8); ax.grid(alpha=.2)
calibration_figure=REPORTS/"elastic_net_walk_forward_calibration.png"; fig.savefig(calibration_figure,dpi=180,bbox_inches="tight"); plt.close(fig)
calibration_bins=pd.concat(calibration_tables,ignore_index=True)
report_html=f'''<!doctype html><html lang="en"><head><meta charset="utf-8"><title>Elastic Net walk-forward benchmark</title><style>
body{{font:15px/1.55 system-ui,sans-serif;margin:30px auto;padding:0 22px;max-width:1200px;color:#203142}}h1,h2{{color:#17425e}}.wrap{{overflow:auto;max-height:600px;margin:14px 0}}table{{border-collapse:collapse;width:100%;font-size:12px}}th,td{{padding:7px;border-bottom:1px solid #d9e1e8;text-align:left;white-space:nowrap}}th{{background:#edf3f7;position:sticky;top:0}}tr:nth-child(even){{background:#f8fafb}}img{{max-width:100%;height:auto}}figure{{margin:22px 0}}figcaption{{font-size:13px;color:#52636e}}.note{{padding:12px 15px;background:#edf6ef;border-left:4px solid #398253}}
</style></head><body><h1>Elastic Net logistic regression benchmark</h1>
<p class="note">Hyperparameters are selected by mean average precision (PR-AUC) across walk-forward validation folds. The operating threshold is selected from pooled out-of-fold validation predictions by balanced accuracy, with sensitivity and specificity tie-breakers. The chronological test is opened only after both decisions are fixed. This matched sample supports relative-risk scores, not population prevalence.</p>
<h2>Selected settings</h2><div class="wrap">{pd.DataFrame([model_configuration]).drop(columns=["training_event_ids","oof_validation_event_ids","test_event_ids","training_constant_predictors"],errors="ignore").to_html(index=False,escape=True,border=0)}</div>
<p>Selected mean validation PR-AUC: {best.mean_validation_pr_auc:.4f}; frozen threshold: {OPERATING_THRESHOLD:.6g}; folds: {html.escape(", ".join(fold_names))}.</p>
<h2>Performance at frozen threshold</h2><div class="wrap">{metrics.to_html(index=False,escape=True,float_format=lambda v:f"{v:.5g}",na_rep="unavailable",border=0)}</div>
<p>Top-risk capture is the share of bankrupt events in the top-scored 1%, 5%, or 10% of rows. Calibration slope and intercept are from logistic recalibration on the logit scores; calibration is descriptive only.</p>
<figure><img src="{performance_figure.name}" alt="ROC and precision-recall curves"><figcaption>Walk-forward OOF validation and final test discrimination.</figcaption></figure>
<figure><img src="{calibration_figure.name}" alt="Reliability diagram"><figcaption>Reliability by quantile bins; small bins are noisy.</figcaption></figure>
<h2>Hyperparameter results</h2><div class="wrap">{tuning.to_html(index=False,escape=True,float_format=lambda v:f"{v:.5g}",border=0)}</div><div class="wrap">{fold_tuning.to_html(index=False,escape=True,float_format=lambda v:f"{v:.5g}",border=0)}</div>
<h2>Calibration bins</h2><div class="wrap">{calibration_bins.to_html(index=False,escape=True,float_format=lambda v:f"{v:.5g}",border=0)}</div>
<p>Each fold used its own training-fitted preprocessing and variance filter. The final model was fitted on the static training split. Test results were not used for feature handling, hyperparameter selection, or threshold choice.</p></body></html>'''


In [5]:
# Save the frozen model and audit outputs.
MODELS.mkdir(parents=True, exist_ok=True)
for path in [fold_ids_path, artifact_path, PROCESSED/"train_matrix.csv", PROCESSED/"train_ids.csv", PROCESSED/"test_matrix.csv", PROCESSED/"test_ids.csv"]:
    if path not in input_hashes: input_hashes[path] = file_hash(path)
final_pipeline.project_model_["input_sha256"] = {str(path.relative_to(ROOT)):digest for path,digest in input_hashes.items()}
joblib.dump(final_pipeline, MODELS/"elastic_net_logistic.joblib", compress=3)
fold_tuning.to_csv(REPORTS/"elastic_net_fold_tuning_results.csv",index=False)
tuning.to_csv(REPORTS/"elastic_net_tuning_results.csv",index=False)
oof_predictions.to_csv(REPORTS/"elastic_net_oof_validation_predictions.csv",index=False)
metrics.to_csv(REPORTS/"elastic_net_metrics.csv",index=False)
coefficients.to_csv(REPORTS/"elastic_net_coefficients.csv",index=False)
predictions.to_csv(REPORTS/"elastic_net_predictions.csv",index=False)
threshold_rows.to_csv(REPORTS/"elastic_net_oof_threshold_candidates.csv",index=False)
(REPORTS/"elastic_net_logistic_report.html").write_text(report_html,encoding="utf-8")
configuration_export={k:v for k,v in model_configuration.items() if k not in {"training_event_ids","oof_validation_event_ids","test_event_ids"}}
(REPORTS/"elastic_net_model_configuration.json").write_text(json.dumps(configuration_export,indent=2,default=str),encoding="utf-8")


980